<a href="https://colab.research.google.com/github/lucas-estanislau/pe-de-meia/blob/Teste-de-otimiza%C3%A7%C3%A3o-da-coleta/pe_de_meia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pé-de-Meia**
### e sua relação com a permanência e o desempenho escolar no ensino médio no Brasil

In [37]:
import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO

In [38]:
mar24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202403"

# Visulizando os dados do mês de março de 2024 para explorar

response = requests.get(mar24, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
  arquivo = zip_ref.namelist()[0]

  with zip_ref.open(arquivo) as csv:
    df = pd.read_csv(csv, sep=";", encoding="latin1")

df

/tmp/ipykernel_39299/2082999193.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv, sep=";", encoding="latin1")


,MÊS FOLHA,MÊS REFERÊNCIA,UF,CÓDIGO MUNICÍPIO SIAFI,NOME MUNICÍPIO,NIS BENEFICIÁRIO,CPF BENEFICIÁRIO,NOME BENEFICIÁRIO,NIS RESPONSÁVEL,CPF RESPONSÁVEL,NOME RESPONSÁVEL,CÓDIGO TIPO INCENTIVO,TIPO INCENTIVO,PERÍODO INCENTIVO,CÓDIGO ETAPA ENSINO,ETAPA ENSINO,DATA DO PAGAMENTO,VALOR PARCELA
0,202403,202403,AC,643,ACRELANDIA,2.126185e+10,***.408.142-**,ALLAN JHONATA DA SILVA HERMOGENES,1.624864e+10,***.338.912-**,IRACILDA HERMOGENES DE SOUZA FERREIRA,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,26/03/2024,"200,00"
1,202403,202403,AC,643,ACRELANDIA,2.120773e+10,***.655.722-**,BIANCA DA SILVA RIBEIRO,1.609672e+10,***.872.892-**,ANTONIA ROSEMIRA SANTOS DA SILVA,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,02/04/2024,"200,00"
2,202403,202403,AC,643,ACRELANDIA,2.380363e+10,***.379.332-**,DAYANE CARVALHO DA SILVA,2.102097e+10,***.323.582-**,ANGELA MARIA CARVALHO,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,03/04/2024,"200,00"
3,202403,202403,AC,643,ACRELANDIA,2.122791e+10,***.703.662-**,DEBORA DANTAS DE LIMA,2.122791e+10,***.703.662-**,DEBORA DANTAS DE LIMA,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,03/04/2024,"200,00"
4,202403,202403,AC,643,ACRELANDIA,2.362701e+10,***.319.892-**,DENNIS BOONE DE CARVALHO,2.093918e+10,***.434.512-**,NELSI BOONE,1,Matrícula,Ano letivo 2024,45,EJA - Educação de Jovens e Adultos,28/03/2024,"200,00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2416247,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.032558e+10,***.326.831-**,ANA LIGIA OLIVEIRA SANTOS,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,02/04/2024,"200,00"
2416248,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.073483e+10,***.793.551-**,JUCINEIA MOREIRA DA SILVA,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,01/04/2024,"200,00"
2416249,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.121601e+10,***.916.451-**,ANA FRANCISCA BEZERRA DA CONCEICAO,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,26/03/2024,"200,00"
2416250,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.063251e+10,***.447.351-**,KEILIANY SOARES BORGES,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,27/03/2024,"200,00"


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2416252 entries, 0 to 2416251
Data columns (total 18 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   MÊS FOLHA               int64  
 1   MÊS REFERÊNCIA          int64  
 2   UF                      object 
 3   CÓDIGO MUNICÍPIO SIAFI  int64  
 4   NOME MUNICÍPIO          object 
 5   NIS BENEFICIÁRIO        float64
 6   CPF BENEFICIÁRIO        object 
 7   NOME BENEFICIÁRIO       object 
 8   NIS RESPONSÁVEL         float64
 9   CPF RESPONSÁVEL         object 
 10  NOME RESPONSÁVEL        object 
 11  CÓDIGO TIPO INCENTIVO   int64  
 12  TIPO INCENTIVO          object 
 13  PERÍODO INCENTIVO       object 
 14  CÓDIGO ETAPA ENSINO     int64  
 15  ETAPA ENSINO            object 
 16  DATA DO PAGAMENTO       object 
 17  VALOR PARCELA           object 
dtypes: float64(2), int64(5), object(11)
memory usage: 331.8+ MB


#Processamento

In [40]:
# Carrega os dados zipados de cada mês do programa dentro do Portal da Transparência
def carregar_mes(url):
  response = requests.get(url, timeout=120)
  response.raise_for_status()

  with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
      arquivo = zip_ref.namelist()[0]

      with zip_ref.open(arquivo) as csv:
          df = pd.read_csv(csv, sep=";", encoding="latin1", usecols=["UF", "VALOR PARCELA"])

  return df

# Converte 'VALOR PARCELA' de object para float
def converter_valor(df):
  df['VALOR PARCELA'] = (
      df['VALOR PARCELA'].str.replace(",", ".", regex=False).astype(float)
  )
  return df

# Coleta os dados do referido mês, trata a coluna de 'VALOR PARCELA' e agrupar por UF. Devolve um df resumido e agrupado
def coletar_mes(url):
  df_mes = carregar_mes(url)
  converter_valor(df_mes)
  df_resumido = df_mes.groupby('UF')['VALOR PARCELA'].sum().reset_index().sort_values('VALOR PARCELA', ascending=False)

  return df_resumido

In [41]:
# Definindo as URL para coletar os arquivos do Portal da Transparência
mar24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202403"
abr24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202404"
mai24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202405"
jun24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202406"
ago24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202408"
set24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202409"
out24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202410"
nov24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202411"
dez24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202412"

# Lista com as variáveis que guardam as URLs
dfs = [mar24,
       abr24,
       mai24,
       jun24,
       ago24,
       set24,
       out24,
       nov24,
       dez24]

# Lista para armazenar os DFs agrupados por UF com soma do valor recebido por cada UF
dfs_resumidos = []

# Coleta os dados de todos os meses resumidos do ano de 2024 e armazena em 'dfs_rsumidos'
for mes in dfs:
  df_resumido = coletar_mes(mes)
  dfs_resumidos.append(df_resumido)

In [42]:
# Lista com os nomes dos meses para salvar
meses = ["mar24", "abr24", "mai24", "jun24", "ago24", "set24", "out24", "nov24", "dez24"
]

# Salva cada DF de acordo com o seu mês
for mes, df in zip(meses, dfs_resumidos):
    df.to_csv(
        f"/content/drive/MyDrive/pe-de-meia/pe-de-meia-data/2024/{mes}_uf.csv",
        index=False
    )